# CICFLOW Stage 8 — XGBoost, cả 4 cấu hình trong một session

Baseline XGBoost: 200 trees / depth10 / min_child_weight3. **MI-RFE-XGB BLOCKED** vì source tham chiếu chưa duyệt RF-RFE policy. Không đọc feature/ranking/best-k/model/scaler của RF.

Chạy setup 1–8, sau đó **Cell 9 xếp hàng và chạy lần lượt cả 4 cấu hình** khi MODE=full. Chỉ xác nhận một lần cho ALL4. Mỗi cấu hình dùng worker/process riêng để giải phóng RAM/GPU. Mặc định preflight; không có fit nếu chưa chủ động chọn full/resume.

Output: `/content/drive/MyDrive/experiment_results/baseline_xgb_<CONFIG>_<RUN_TAG>/`. PROVISIONAL_RESEARCH, OFFICIAL_BLOCKED, capture/session UNKNOWN; không có tuyên bố training PASS trước khi bạn chạy.


## Cell 1 — Mount Drive

Giữ dataset và kết quả trên MyDrive; không remount bắt buộc.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(DRIVE_ROOT)


## Cell 2 — Dependencies và runtime

Chỉ hiển thị environment, không tự pip install hoặc đổi backend. Profile nguồn cố định; thiếu/sai version sẽ BLOCKED tại preflight.


In [ ]:
import importlib.metadata as metadata
import json, os, platform, shutil, sys
MODE = "preflight"
EXPERIMENT = "baseline_xgb"
BACKEND = "cpu"
RUN_TAG = "run01"
reference = {"numpy":"2.2.6", "pandas":"2.3.3", "scipy":"1.15.3", "scikit-learn":"1.7.2",
             "joblib":"1.5.3", "psutil":"5.9.5", "xgboost":"3.4.1"}
print("Python:", sys.version, platform.platform(), "CPUs:", os.cpu_count())
for name in (*reference, "pyarrow", "nvidia-ml-py"):
    try:
        print(name, metadata.version(name), "reference:", reference.get(name, "record installed"))
    except metadata.PackageNotFoundError:
        print(name, "MISSING")
print("No automatic package installation; no fit.")


## Cell 3 — Clone/import đúng commit

Dán SHA thật của commit đã push. Chỉ clone vào thư mục Colab mới theo SHA; checkout có sẵn phải đúng commit và sạch, không pull/reset source đang sửa.


In [ ]:
import subprocess
REPO_URL = "https://github.com/kasum00/RFforNetwork.git"  # source URL from reference notebook
APPROVED_COMMIT = ""  #@param {type:"string"}
if len(APPROVED_COMMIT) != 40 or any(c not in "0123456789abcdefABCDEF" for c in APPROVED_COMMIT):
    raise ValueError("Paste the exact 40-hex commit containing code_CICFLOW/stage8_xgboost")
APPROVED_COMMIT = APPROVED_COMMIT.lower()
REPO = Path("/content") / ("CICFLOW_STAGE8_" + APPROVED_COMMIT[:12])
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", "--", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", APPROVED_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", APPROVED_COMMIT], check=True)
actual = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
if actual.lower() != APPROVED_COMMIT.lower() or dirty:
    raise RuntimeError({"head": actual, "dirty": dirty, "message": "Preserve existing checkout; use the approved clean source."})
SOURCE_ROOT = REPO / "code_CICFLOW"
required = ("stage8_xgboost/src/runner.py", "stage8_xgboost/src/checkpoint_manager.py",
            "stage8_xgboost/configs/xgboost_contract.json",
            "colab/cicflow_reference_pipeline.py", "stage4_mi/mi_selection.py", "stage5_rfe/rfe_selection.py")
for relative in required:
    if not (SOURCE_ROOT / relative).is_file():
        raise FileNotFoundError(SOURCE_ROOT / relative)
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE_ROOT))
from stage8_xgboost.src import runner, common, drive_artifact_manager
print("Source:", SOURCE_ROOT, "\nCommit:", actual, "\nSHA256:", common.snapshot(SOURCE_ROOT)["sha256"])


## Cell 4 — Paths và bốn cấu hình

Đúng root CICFLOW; không WINDOWS_CPLUS, không gộp bốn bảng. CONFIGS luôn là bốn cấu hình riêng.


In [ ]:
DATASET_ROOT = DRIVE_ROOT / "CICFLOWMETER_FEATURE_CSV"
RESULT_ROOT = DRIVE_ROOT / "experiment_results"
CONFIGS = common.CONFIGS
for config in CONFIGS:
    folder = DATASET_ROOT / config
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    for name in ("window_integrity.json", "window_feature_schema.json"):
        if not (folder / name).is_file():
            raise FileNotFoundError(f"Upload existing small sidecar: {folder/name}")
    print(config, "FOUND")
SETUP = {
    "source": str(SOURCE_ROOT), "dataset": str(DATASET_ROOT), "result_root": str(RESULT_ROOT),
    "configs": list(CONFIGS), "experiment": EXPERIMENT, "mode": MODE, "backend": BACKEND,
    "run_tag": RUN_TAG, "resume": False, "git_commit": APPROVED_COMMIT, "stop_on_error": False,
}
print("Result root:", RESULT_ROOT)
print("RF results are protected. No Parquet is copied.")


## Cell 5 — Dataset/feature contract

Footers và sidecars của cả 4 cấu hình; kiểm tra 380 feature, order, split counts, proxy-label histogram. Không đọc Train/Validation/Test values.


In [ ]:
METADATA = runner.run_all(SETUP, action="metadata")
for row in METADATA["rows"]:
    result = row.get("result", {})
    print(row["config"], row["program_status"], result.get("binary_label_status"), result.get("label_mapping_provenance"))
if METADATA["status"] != "PASS":
    raise RuntimeError("Metadata BLOCKED. Inspect logs inside each run; do not regenerate/split/sample the dataset.")


## Cell 6 — Resource preflight

CPU là mặc định. CUDA phải chọn rõ, có build/GPU/VRAM phù hợp; không CPU fallback. Budget dưới đây là chính sách admission CICFLOW cần bạn xem, không phải scientific hyperparameters và không phải benchmark.


In [ ]:
# Explicit CICFLOW budget; original C+ notebook had 44GiB/40GiB because it used another dataset/cache.
BUDGET = {
    "min_start_available_ram_bytes": 8 * 2**30,
    "min_runtime_available_ram_bytes": 4 * 2**30,
    "max_tree_rss_bytes": 40 * 2**30,
    "min_free_disk_bytes": 4 * 2**30,
    "max_elapsed_seconds": 43200,
    "min_gpu_free_bytes": 4 * 2**30,
}
SETUP["budget"] = BUDGET
RESOURCES = runner.run_all(SETUP, action="resources")
for row in RESOURCES["rows"]:
    print(row["config"], row["program_status"], row.get("result", {}).get("resources", row.get("result")))
if RESOURCES["status"] != "PASS":
    print("Resource/version preview BLOCKED. Full admission will block fit; inspect logs. You may continue setup to choose verify/resume.")


## Cell 7 — Unit tests và checkpoint status

Bạn chủ động chạy fixture/mock tests; không training dataset thật. Hiển thị durable state, không tự sửa checkpoint. Tests được bind source/environment.


In [ ]:
TEST_RUN = runner.launch("tests", {**SETUP, "config": CONFIGS[0]})
UNIT_TEST_EVIDENCE = TEST_RUN["result"]
print(json.dumps(UNIT_TEST_EVIDENCE, indent=2))
if TEST_RUN["exit_code"] != 0 or TEST_RUN["program_status"] != "PASS":
    raise RuntimeError("Unit tests FAIL/no tests/skips; full training BLOCKED")
for config in CONFIGS:
    directory = common.output_path({**SETUP, "config": config})
    protocol = directory / "checkpoints/booster/protocol_state.json"
    print(config, common.read(protocol) if protocol.is_file() else "NOT_STARTED")
print("Fixture PASS does not establish official scientific validity.")


## Cell 8 — Chọn experiment và ALL4

Đây là cấu hình chính. Chọn full để chạy cả bốn, resume để tiếp tục cả batch cũ, verify để chỉ kiểm tra. Không cần chọn từng FW/FS. Dùng run02 nếu muốn experiment mới.


In [ ]:
MODE = "preflight"  #@param ["preflight", "full", "resume", "verify"]
EXPERIMENT = "baseline_xgb"  #@param ["baseline_xgb", "mi_rfe_xgb"]
BACKEND = "cpu"  #@param ["cpu", "cuda"]
RUN_TAG = "run01"  #@param {type:"string"}
STOP_ON_ERROR = False  #@param {type:"boolean"}
RESUME = MODE == "resume"
REQUEST = {
    **SETUP, "mode": MODE, "experiment": EXPERIMENT, "backend": BACKEND,
    "run_tag": RUN_TAG, "resume": RESUME, "stop_on_error": STOP_ON_ERROR,
    "budget": BUDGET, "unit_tests": UNIT_TEST_EVIDENCE,
}
runner.validate_batch(REQUEST)
for CONFIG in CONFIGS:
    RUN_NAME = common.output_path({**REQUEST, "config": CONFIG}).name
    print(CONFIG, "->", RESULT_ROOT / RUN_NAME)
print("One confirmation:", common.confirmation(REQUEST))
print("Sequential workers: 1, not four simultaneous fits.")
print("MI-RFE-XGB remains BLOCKED; no RF feature selection artifacts are read.")


## Cell 9 — Một lần bấm: preflight/full/resume/verify cả 4

Đây là cell chạy chính. Full/resume hỏi xác nhận ALL4 một lần rồi tự xử lý 4 cấu hình tuần tự. Cấu hình hoàn tất được verify/skip; lỗi một cấu hình vẫn tiếp tục cấu hình khác khi STOP_ON_ERROR=False. Test mỗi cấu hình chỉ một lần.


In [ ]:
if MODE in ("full", "resume"):
    expected = common.confirmation(REQUEST)
    answer = input("Gõ chính xác để chạy cả 4 cấu hình với budget đã xem:\n" + expected + "\n").strip()
    if answer != expected:
        raise PermissionError("ALL4 confirmation mismatch; no training started")
    REQUEST["confirmation"] = answer
BATCH_RESULT = runner.run_all(REQUEST)
for row in BATCH_RESULT["rows"]:
    print(row["config"], row["program_status"], "exit:", row.get("exit_code"),
          row.get("result", {}).get("directory", row.get("error", "")))
print("Batch technical status:", BATCH_RESULT["status"])
print("Scientific status remains PROVISIONAL_RESEARCH / OFFICIAL_BLOCKED.")


## Cell 10 — Resume an toàn sau restart

Không khởi chạy thêm experiment ở cell này. Sau restart chạy setup lại với đúng commit/environment/run tag/backend/budget; Cell 8 chọn resume, rồi Cell 9 xử lý cả bốn. Fit tiếp từ round đã persist; không hứa bitwise tương đương CPU/CUDA hoặc giữa các version.


In [ ]:
print("Resume:")
print("1. Same APPROVED_COMMIT, source/library versions, BACKEND, BUDGET and RUN_TAG.")
print("2. Cell 8: MODE='resume'; RUN_TAG stays the existing run tag.")
print("3. Cell 9: confirm RESUME ALL4 once.")
print("Completed runs are verified/skipped; not-started configs start fresh; incomplete configs use verified checkpoints.")
print("Checkpointed stages: validation_fit and final_refit; 10-round boundaries, total exactly200 each.")
print("TEST_EVALUATION_REQUESTED/TEST_STATE_UNKNOWN blocks another Test evaluation.")
print("No checkpoint/result/RF artifact is deleted to force resume.")


## Cell 11 — Final refit và baseline selection

Đọc state đã lưu. Baseline không có MI/RFE/k-search: NOT_APPLICABLE, đủ 380 feature. Model/scaler cuối fit Train+Validation sau freeze, từng cấu hình riêng.


In [ ]:
for CONFIG in CONFIGS:
    directory = common.output_path({**REQUEST, "config": CONFIG})
    selection = directory / "feature_selection.json"
    final = directory / "checkpoints/final_refit_artifacts.json"
    print(CONFIG)
    if selection.exists():
        item = common.read(selection)
        print("Selection:", item["status"], "features:", len(item["selected_features"]), "best_k:", item["best_k"])
    if final.exists():
        item = common.read(final)
        print("Final refit rows:", item["rows"], "completed rounds:", item["completed_rounds"])
    else:
        print("No completed final refit yet")


## Cell 12 — Test-once evidence

Chỉ đọc metrics/protocol đã lưu; cell này không load dataset/model và không predict Test.


In [ ]:
for CONFIG in CONFIGS:
    directory = common.output_path({**REQUEST, "config": CONFIG})
    protocol = directory / "checkpoints/booster/protocol_state.json"
    metric_file = directory / "metrics/test_metrics.json"
    print(CONFIG)
    if protocol.exists():
        state = common.read(protocol)
        print("Test state:", state["test_state"], "calls:", state["test_evaluation_calls"])
    if metric_file.exists():
        item = common.read(metric_file)
        print("Saved Test rows:", item["rows"], "metrics:", item["metrics"])
    else:
        print("No completed Test metrics; no placeholder PASS")


## Cell 13 — Export và artifact verification

Artifact được ghi trực tiếp lên Drive trong lúc chạy; không cần copy Parquet. Chỉ verify config đã có RUN_COMPLETE. Worker exit0 và hash validation phải xong trước marker.


In [ ]:
VERIFIED = []
for CONFIG in CONFIGS:
    directory = common.output_path({**REQUEST, "config": CONFIG})
    if (directory / "RUN_COMPLETE.json").exists():
        result = drive_artifact_manager.verify(directory)
        VERIFIED.append(result)
        print(CONFIG, result["status"], directory)
    else:
        print(CONFIG, "INCOMPLETE/NOT_STARTED; inspect", directory / "logs/execution.log")
print("No dataset values, model load, fit or Test prediction in this cell.")


## Cell 14 — Bảng kết quả cả bốn

Summary từ JSON đã verify; không suy ra scientific PASS từ exit0. Nếu thiếu cấu hình thì hiển thị INCOMPLETE.


In [ ]:
ROWS = []
for CONFIG in CONFIGS:
    directory = common.output_path({**REQUEST, "config": CONFIG})
    if (directory / "RUN_COMPLETE.json").exists():
        saved = drive_artifact_manager.verify(directory)
        ROWS.append({"config": CONFIG, "run_name": directory.name, "status": "VERIFIED_COMPLETE",
                     **saved["metrics"], "scientific_status": "PROVISIONAL_RESEARCH"})
    else:
        ROWS.append({"config": CONFIG, "run_name": directory.name, "status": "INCOMPLETE_OR_NOT_STARTED"})
import pandas as pd
frame = pd.DataFrame(ROWS)
display(frame[[c for c in ("config", "run_name", "status", "accuracy", "precision_attack", "recall_attack",
                           "f1_attack", "macro_f1", "balanced_accuracy", "scientific_status") if c in frame]])
print("All results:", RESULT_ROOT)
print("Per config: logs/execution.log, preflight.json, run_manifest.json, configuration.json,")
print("feature_selection.json, checkpoints/, models/, metrics/, reports/, RUN_COMPLETE.json")
print("Send preflight.json, run_manifest.json, metrics/*.json, reports/BATCH_*.json and execution.log.")
print("Do not send Parquet or large models unless needed.")
